### List of Experiments in this notebook
#### Models considered : gpt-4.1-mini, llama 3.2-1b-it
- asking questions one by one, as the outlines framework doesn't allow for multiple answers to be returned at once
- asking both normal hexaco and paraphrased hexaco questions
- providing both normal likert scale and inverted likert scale
- running the experiments on base model with basic information from the persona definition, asking the hexaco questions directly in the likert scale
    - giving an option to refuse to answer
    - not giving an option to refuse to answer
- evaluating refusal rate
- checking paraphrase reliability
- calculating trait wise hexaco scores and refusal rates

In [4]:
import torch as t
import outlines
from transformers import AutoTokenizer, AutoModelForCausalLM
from pydantic import BaseModel
from typing import Literal
from enum import Enum
import yaml
import pandas as pd
import numpy as np
import sys
import os
import json
from openai import OpenAI
sys.path.append("../")
from src.utils import inverse_likert, list_to_str
device = t.device("cuda" if t.cuda.is_available() else "cpu")

In [5]:
class OpenaiResponse(BaseModel):
    response: str

In [30]:
with open('../configs/generation_config.yaml', 'r') as file:
    generation_config = yaml.safe_load(file)
    
with open('../psychometric_tests/hexaco_100_questions.yaml', 'r') as file:
    question_list = yaml.safe_load(file)
    
with open('../psychometric_tests/paraphrased_hexaco_100_questions.yaml', 'r') as file:
    paraphrased_question_list = yaml.safe_load(file)
    
with open('../psychometric_tests/hexaco_100_eval.yaml', 'r') as file:
    hexaco_eval = yaml.safe_load(file)
    
with open('../configs/personas.yaml', 'r') as file:
    personas = yaml.safe_load(file)

In [7]:
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
model = outlines.from_transformers(
    AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map=device),
    AutoTokenizer.from_pretrained(MODEL_NAME)
)

In [8]:
openai_model_name = "gpt-4.1-mini"
openai_model = outlines.from_openai(OpenAI(), openai_model_name)

In [9]:
NO_ANSWER = "Do not wish to answer"
likert_scale = generation_config['likert_scale'].copy()
likert_scale.append(NO_ANSWER)

In [10]:
inverted_likert = inverse_likert(generation_config['likert_scale'].copy())
inverted_likert.append(NO_ANSWER)

In [11]:
likert_scale_without_no = generation_config['likert_scale'].copy()
inverted_likert_without_no = inverse_likert(generation_config['likert_scale'].copy())

In [31]:
base_text = personas['customer_service']['base_text']
persona = personas['customer_service']['personas'][0]

In [36]:
", ".join([f"{key} : {persona[key]}" for key in persona if key != "Summary"])

'Age : 42, Location : Lagos, Nigeria, Background : Former emergency dispatcher'

In [44]:
base_hexaco_template = outlines.Template.from_string("""
<|im_start>user
Task: Answer the below questions:

{{ text }}

Answer the question as either {{ likert_scale }}.
<|im_end>
<|im_start>assistant
""")

persona_hexaco_template = outlines.Template.from_string("""
<|im_start>user
{{base_text}} with following attributes :

{{attributes}}

Task: Answer the below questions:

{{ text }}

Answer the question as either {{ likert_scale }}.
<|im_end>
<|im_start>assistant
""")


In [47]:
print(base_hexaco_template(text = question_list[0], likert_scale = likert_scale, base_text = base_text, attributes = ", ".join([f"{key} : {persona[key]}" for key in persona if key != "Summary"])))

<|im_start>user
Task: Answer the below questions:

I would be quite bored by a visit to an art gallery.

Answer the question as either ['Strongly Disagree', 'Disagree', 'Neutral', 'Agree', 'Strongly Agree', 'Do not wish to answer'].
<|im_end>
<|im_start>assistant


In [45]:
prompt = persona_hexaco_template(text = question_list[0], likert_scale = likert_scale, base_text = base_text, attributes = ", ".join([f"{key} : {persona[key]}" for key in persona if key != "Summary"]))
print(prompt)

<|im_start>user
You are a customer service professional. with following attributes :

Age : 42, Location : Lagos, Nigeria, Background : Former emergency dispatcher

Task: Answer the below questions:

I would be quite bored by a visit to an art gallery.

Answer the question as either ['Strongly Disagree', 'Disagree', 'Neutral', 'Agree', 'Strongly Agree', 'Do not wish to answer'].
<|im_end>
<|im_start>assistant


In [61]:
def local_generation(hexaco_template, model, question, likert_scale, persona_str=None, persona_base_text=None):
    prompt = hexaco_template(text=question, likert_scale = ", ".join(likert_scale), base_text = persona_base_text, attributes = persona_str)
    answer = model(
                    prompt,
                    Literal[*likert_scale]
            )
    return answer

def openai_generation(hexaco_template, model, question, likert_scale, persona_str=None, persona_base_text=None):
    prompt = hexaco_template(text=question, likert_scale = ", ".join(likert_scale), base_text = persona_base_text, attributes = persona_str)
    prompt = f"{prompt}, use the json format."
    
    answer = openai_model(prompt, OpenaiResponse)
    return json.loads(answer)['response']
    

def generate_answers(generation_function, model, question_list, likert_scale, job_title=None):
    
    if job_title:
        answers = []
        hexaco_template = persona_hexaco_template
        base_text = personas[job_title]['base_text']
        persona_list = personas[job_title]['personas']
        for persona in persona_list:
            persona_answer = []
            persona_str = ", ".join([f"{key} : {persona[key]}" for key in persona if key != "Summary"])
            for question in question_list:
                answer = generation_function(hexaco_template, model, question, likert_scale, persona_str, base_text)
                persona_answer.append(answer)
            persona_dict = {}
            persona_dict['persona'] = persona_str
            persona_dict['answers'] = persona_answer
            answers.append(persona_dict)        
        
    else:
        hexaco_template = base_hexaco_template  
        answers = []
        for question in question_list:
            answer = generation_function(hexaco_template, model, question, likert_scale)
            answers.append(answer)
    
    return answers

In [53]:
personas['customer_service']

{'base_text': 'You are a customer service professional.',
 'personas': [{'Summary': 'Calm Crisis Handler',
   'Age': 42,
   'Location': 'Lagos, Nigeria',
   'Background': 'Former emergency dispatcher'},
  {'Summary': 'Empathetic Listener',
   'Age': 29,
   'Location': 'Mumbai, India',
   'Background': 'Psychology graduate turned support specialist'}]}

In [54]:
def write_to_json(file, file_path):
    with open(file_path, 'w') as f:
        json.dump(file, f)
        
def read_json(file_path):
    with open(file_path, "r") as f:
        file = json.load(f)
    return file

In [55]:
print(prompt)

<|im_start>user
You are a customer service professional. with following attributes :

Age : 42, Location : Lagos, Nigeria, Background : Former emergency dispatcher

Task: Answer the below questions:

I would be quite bored by a visit to an art gallery.

Answer the question as either ['Strongly Disagree', 'Disagree', 'Neutral', 'Agree', 'Strongly Agree', 'Do not wish to answer'].
<|im_end>
<|im_start>assistant


In [56]:
job_title = "customer_service"

### Normal Questions, Normal Likert

In [64]:
normal_hexaco_answers_gpt_41_mini = generate_answers(openai_generation, openai_model, question_list, likert_scale, job_title)
write_to_json(normal_hexaco_answers_gpt_41_mini, os.path.join("persona_basic_info_experiment_results","normal_hexaco_answers_gpt_41_mini.json"))

In [65]:
normal_hexaco_answers_without_no_gpt_41_mini = generate_answers(openai_generation, openai_model, question_list, likert_scale_without_no, job_title)
write_to_json(normal_hexaco_answers_without_no_gpt_41_mini, os.path.join("persona_basic_info_experiment_results","normal_hexaco_answers_without_no_gpt_41_mini.json"))

In [62]:
normal_hexaco_answers_llama_3_2b_it = generate_answers(local_generation, model, question_list, likert_scale, job_title)
write_to_json(normal_hexaco_answers_llama_3_2b_it, os.path.join("persona_basic_info_experiment_results","normal_hexaco_answers_llama_3_2b_it.json"))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

In [63]:
normal_hexaco_answers_without_no_llama_3_2b_it = generate_answers(local_generation, model, question_list, likert_scale_without_no, job_title)
write_to_json(normal_hexaco_answers_without_no_llama_3_2b_it, os.path.join("persona_basic_info_experiment_results","normal_hexaco_answers_without_no_llama_3_2b_it.json"))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

### Normal Questions, Inverse_Likert

In [69]:
normal_hexaco_inverted_likert_answers_gpt_41_mini = generate_answers(openai_generation, openai_model, question_list, inverted_likert, job_title)
write_to_json(normal_hexaco_inverted_likert_answers_gpt_41_mini, os.path.join("persona_basic_info_experiment_results","normal_hexaco_inverted_likert_answers_gpt_41_mini.json"))

In [70]:
normal_hexaco_inverted_likert_without_no_answers_gpt_41_mini = generate_answers(openai_generation, openai_model, question_list, inverted_likert_without_no, job_title)
write_to_json(normal_hexaco_inverted_likert_without_no_answers_gpt_41_mini, os.path.join("persona_basic_info_experiment_results","normal_hexaco_inverted_likert_without_no_answers_gpt_41_mini.json"))

In [67]:
normal_hexaco_inverted_likert_answers_llama_3_2b_it = generate_answers(local_generation, model, question_list, inverted_likert, job_title)
write_to_json(normal_hexaco_inverted_likert_answers_llama_3_2b_it, os.path.join("persona_basic_info_experiment_results","normal_hexaco_inverted_likert_answers_llama_3_2b_it.json"))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

In [68]:
normal_hexaco_inverted_likert_without_no_answers_llama_3_2b_it = generate_answers(local_generation, model, question_list, inverted_likert_without_no, job_title)
write_to_json(normal_hexaco_inverted_likert_without_no_answers_llama_3_2b_it, os.path.join("persona_basic_info_experiment_results","normal_hexaco_inverted_likert_without_no_answers_llama_3_2b_it.json"))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

### Paraphrase Questions, Normal Likert

In [71]:
paraphrase_hexaco_answers_gpt_41_mini = generate_answers(openai_generation, openai_model, paraphrased_question_list, likert_scale, job_title)
write_to_json(paraphrase_hexaco_answers_gpt_41_mini, os.path.join("persona_basic_info_experiment_results","paraphrase_hexaco_answers_gpt_41_mini.json"))

In [72]:
paraphrase_hexaco_answers_without_no_gpt_41_mini = generate_answers(openai_generation, openai_model, paraphrased_question_list, likert_scale_without_no, job_title)
write_to_json(paraphrase_hexaco_answers_without_no_gpt_41_mini, os.path.join("persona_basic_info_experiment_results","paraphrase_hexaco_answers_without_no_gpt_41_mini.json"))

In [73]:
paraphrase_hexaco_answers_llama_3_2b_it = generate_answers(local_generation, model, paraphrased_question_list, likert_scale, job_title)
write_to_json(paraphrase_hexaco_answers_llama_3_2b_it, os.path.join("persona_basic_info_experiment_results","paraphrase_hexaco_answers_llama_3_2b_it.json"))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

In [74]:
paraphrase_hexaco_answers_without_no_llama_3_2b_it = generate_answers(local_generation, model, paraphrased_question_list, likert_scale_without_no, job_title)
write_to_json(paraphrase_hexaco_answers_without_no_llama_3_2b_it, os.path.join("persona_basic_info_experiment_results","paraphrase_hexaco_answers_without_no_llama_3_2b_it.json"))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

### Paraphrase Questions, Inverted Likert

In [75]:
paraphrase_hexaco_inverted_likert_answers_gpt_41_mini = generate_answers(openai_generation, openai_model, paraphrased_question_list, inverted_likert, job_title)
write_to_json(paraphrase_hexaco_inverted_likert_answers_gpt_41_mini, os.path.join("persona_basic_info_experiment_results","paraphrase_hexaco_inverted_likert_answers_gpt_41_mini.json"))

In [76]:
paraphrase_hexaco_inverted_likert_without_no_answers_gpt_41_mini = generate_answers(openai_generation, openai_model, paraphrased_question_list, inverted_likert_without_no, job_title)
write_to_json(paraphrase_hexaco_inverted_likert_without_no_answers_gpt_41_mini, os.path.join("persona_basic_info_experiment_results","paraphrase_hexaco_inverted_likert_without_no_answers_gpt_41_mini.json"))

In [77]:
paraphrase_hexaco_inverted_likert_answers_llama_3_2b_it = generate_answers(local_generation, model, paraphrased_question_list, inverted_likert, job_title)
write_to_json(paraphrase_hexaco_inverted_likert_answers_llama_3_2b_it, os.path.join("persona_basic_info_experiment_results","paraphrase_hexaco_inverted_likert_answers_llama_3_2b_it.json"))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

In [78]:
paraphrase_hexaco_inverted_likert_without_no_answers_llama_3_2b_it = generate_answers(local_generation, model, paraphrased_question_list, inverted_likert_without_no, job_title)
write_to_json(paraphrase_hexaco_inverted_likert_without_no_answers_llama_3_2b_it, os.path.join("persona_basic_info_experiment_results","paraphrase_hexaco_inverted_likert_without_no_answers_llama_3_2b_it.json"))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

### Evaluation

In [321]:
def get_refusal_rate(answers):  
    answers = pd.Series(answers)
    refusal_answers = answers[answers == "Do not wish to answer"]
    return len(refusal_answers)/len(answers)

In [322]:
refusal_rate_dict = {}
for filename in os.listdir("base_experiment_results"):
    answers = read_json(os.path.join("base_experiment_results",filename))
    refusal_rate = get_refusal_rate(answers)
    refusal_rate_dict[filename.split(".")[0]] = refusal_rate

In [323]:
dict(sorted(refusal_rate_dict.items()))

{'normal_hexaco_answers_gpt_41_mini': 0.01,
 'normal_hexaco_answers_llama_3_2b_it': 0.09,
 'normal_hexaco_answers_without_no_gpt_41_mini': 0.0,
 'normal_hexaco_answers_without_no_llama_3_2b_it': 0.0,
 'normal_hexaco_inverted_likert_answers_gpt_41_mini': 0.02,
 'normal_hexaco_inverted_likert_answers_llama_3_2b_it': 0.11,
 'normal_hexaco_inverted_likert_without_no_answers_gpt_41_mini': 0.0,
 'normal_hexaco_inverted_likert_without_no_answers_llama_3_2b_it': 0.0,
 'paraphrase_hexaco_answers_gpt_41_mini': 0.02,
 'paraphrase_hexaco_answers_llama_3_2b_it': 0.14,
 'paraphrase_hexaco_answers_without_no_gpt_41_mini': 0.0,
 'paraphrase_hexaco_answers_without_no_llama_3_2b_it': 0.0,
 'paraphrase_hexaco_inverted_likert_answers_gpt_41_mini': 0.0,
 'paraphrase_hexaco_inverted_likert_answers_llama_3_2b_it': 0.16,
 'paraphrase_hexaco_inverted_likert_without_no_answers_gpt_41_mini': 0.0,
 'paraphrase_hexaco_inverted_likert_without_no_answers_llama_3_2b_it': 0.0}

In [348]:
def calculate_hexaco_score(trait, subtrait, answers, likert_scale = generation_config['likert_scale']):
    answer_dict = {}
    answers = pd.Series(answers)
    subtrait_dict = hexaco_eval[trait][subtrait]
    indices = [idx - 1 for idx in subtrait_dict['indices']]
    trait_answers = answers[indices]
    refused_answers = trait_answers[trait_answers == 'Do not wish to answer']
    non_refused_answers = trait_answers[trait_answers != 'Do not wish to answer']
    answer_indices = [likert_scale.index(answer) for answer in non_refused_answers]
    true_answer_indices = [6-idx if reverse else idx for idx,reverse in zip(answer_indices, subtrait_dict['reverse'])]
    
    answer_dict['answer_indices'] = answer_indices
    answer_dict['true_answer_indices'] = true_answer_indices
    answer_dict['n_answered_questions'] = len(true_answer_indices)
    answer_dict['n_refused_questions'] = len(refused_answers)
    answer_dict['trait'] = trait
    answer_dict['subtrait'] = subtrait
    answer_dict['subtrait_score'] = np.round(np.mean(true_answer_indices).item(),3)
    return answer_dict

In [354]:
def get_all_stats(filename, hexaco_eval, likert_scale=generation_config['likert_scale']):
    subtrait_hexaco_scores = []
    trait_hexaco_scores = []
    answers = read_json(os.path.join("base_experiment_results",filename))
    for trait in hexaco_eval.keys():
        trait_hexaco_score = {}
        trait_hexaco_score['trait'] = trait
        trait_hexaco_score['true_answer_indices'] = []
        trait_hexaco_score['n_answered_questions'] = 0
        trait_hexaco_score['n_refused_questions'] = 0
        for subtrait in hexaco_eval[trait].keys():
            subtrait_hexaco_score = calculate_hexaco_score(trait, subtrait, answers, likert_scale)
            trait_hexaco_score['true_answer_indices'].extend(subtrait_hexaco_score['true_answer_indices'])
            trait_hexaco_score['n_answered_questions'] += subtrait_hexaco_score['n_answered_questions']
            trait_hexaco_score['n_refused_questions'] += subtrait_hexaco_score['n_refused_questions']
            if "inverted" in filename:
                subtrait_hexaco_score['likert_scale'] = "inverse"
            else:
                subtrait_hexaco_score['likert_scale'] = "normal"
            if "paraphrase" in filename:
                subtrait_hexaco_score['paraphrase'] = "paraphrase"
            else:
                subtrait_hexaco_score['paraphrase'] = "normal"
            if "without_no" in filename:
                subtrait_hexaco_score['refusal_allowed'] = "No Refusal"
            else:
                subtrait_hexaco_score['refusal_allowed'] = "Refusal"
            if "llama" in filename:
                subtrait_hexaco_score['model'] = "llama_3.2_1b_it"
            else:
                subtrait_hexaco_score['model'] = "gpt_4.1_mini"
                
            subtrait_hexaco_scores.append(subtrait_hexaco_score)
        trait_hexaco_score['trait_score'] = np.round(np.mean(trait_hexaco_score['true_answer_indices']).item(),3)
        if "inverted" in filename:
            trait_hexaco_score['likert_scale'] = "inverse"
        else:
            trait_hexaco_score['likert_scale'] = "normal"
        if "paraphrase" in filename:
            trait_hexaco_score['paraphrase'] = "paraphrase"
        else:
            trait_hexaco_score['paraphrase'] = "normal"
        if "without_no" in filename:
            trait_hexaco_score['refusal_allowed'] = "No Refusal"
        else:
            trait_hexaco_score['refusal_allowed'] = "Refusal"
        if "llama" in filename:
            trait_hexaco_score['model'] = "llama_3.2_1b_it"
        else:
            trait_hexaco_score['model'] = "gpt_4.1_mini"
        trait_hexaco_scores.append(trait_hexaco_score)
    return subtrait_hexaco_scores,trait_hexaco_scores

In [355]:
subtrait_stat_df = pd.DataFrame()
trait_stat_df = pd.DataFrame()
for filename in os.listdir("base_experiment_results"):
    subtrait_hexaco_scores,trait_hexaco_scores = get_all_stats(filename, hexaco_eval)
    subtrait_df = pd.DataFrame(subtrait_hexaco_scores)
    trait_df = pd.DataFrame(trait_hexaco_scores)
    subtrait_stat_df = pd.concat([subtrait_stat_df,subtrait_df], axis = 0)
    trait_stat_df = pd.concat([trait_stat_df,trait_df], axis = 0)
subtrait_stat_df.reset_index(inplace=True, drop=True)
trait_stat_df.reset_index(inplace=True, drop=True)

In [356]:
gpt_subtrait_stat_df = subtrait_stat_df[subtrait_stat_df['model'] == "gpt_4.1_mini"]
llama_subtrait_stat_df = subtrait_stat_df[subtrait_stat_df['model'] == "llama_3.2_1b_it"]

gpt_trait_stat_df = trait_stat_df[trait_stat_df['model'] == "gpt_4.1_mini"]
llama_trait_stat_df = trait_stat_df[trait_stat_df['model'] == "llama_3.2_1b_it"]

In [358]:
gpt_subtrait_stat_df.loc[:,'refusal_rate'] = gpt_subtrait_stat_df.loc[:,'n_refused_questions']/(gpt_subtrait_stat_df.loc[:,'n_answered_questions'] + gpt_subtrait_stat_df.loc[:,'n_refused_questions'])
llama_subtrait_stat_df.loc[:,'refusal_rate'] = llama_subtrait_stat_df.loc[:,'n_refused_questions']/(llama_subtrait_stat_df.loc[:,'n_answered_questions'] + llama_subtrait_stat_df.loc[:,'n_refused_questions'])

gpt_trait_stat_df.loc[:,'refusal_rate'] = gpt_trait_stat_df.loc[:,'n_refused_questions']/(gpt_trait_stat_df.loc[:,'n_answered_questions'] + gpt_trait_stat_df.loc[:,'n_refused_questions'])
llama_trait_stat_df.loc[:,'refusal_rate'] = llama_trait_stat_df.loc[:,'n_refused_questions']/(llama_trait_stat_df.loc[:,'n_answered_questions'] + llama_trait_stat_df.loc[:,'n_refused_questions'])

/var/folders/9b/k67tngbx13jgzw5kjx9_d0tc0000gn/T/ipykernel_27949/2755542078.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gpt_subtrait_stat_df.loc[:,'refusal_rate'] = gpt_subtrait_stat_df.loc[:,'n_refused_questions']/(gpt_subtrait_stat_df.loc[:,'n_answered_questions'] + gpt_subtrait_stat_df.loc[:,'n_refused_questions'])
/var/folders/9b/k67tngbx13jgzw5kjx9_d0tc0000gn/T/ipykernel_27949/2755542078.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  llama_subtrait_stat_df.loc[:,'refusal_rate'] = llama_sub

In [360]:
gpt_subtrait_stat_pivot_df = gpt_subtrait_stat_df.pivot(values = ['refusal_rate','subtrait_score',], columns=['paraphrase', 'likert_scale','refusal_allowed'], index=['trait','subtrait'])

gpt_trait_stat_pivot_df = gpt_trait_stat_df.pivot(values = ['refusal_rate','trait_score',], columns=['paraphrase', 'likert_scale','refusal_allowed'], index=['trait'])

In [361]:
llama_subtrait_stat_pivot_df = llama_subtrait_stat_df.pivot(values = ['refusal_rate','subtrait_score',], columns=['paraphrase', 'likert_scale','refusal_allowed'], index=['trait','subtrait'])

llama_trait_stat_pivot_df = llama_trait_stat_df.pivot(values = ['refusal_rate','trait_score',], columns=['paraphrase', 'likert_scale','refusal_allowed'], index=['trait'])

In [365]:
gpt_trait_stat_pivot_df

refusal_rate                                           \
paraphrase               paraphrase     normal            paraphrase           
likert_scale                 normal     normal    inverse     normal inverse   
refusal_allowed             Refusal No Refusal No Refusal No Refusal Refusal   
trait                                                                          
agreeableness                 0.000        0.0        0.0        0.0     0.0   
altruism                      0.000        0.0        0.0        0.0     0.0   
conscientiousness             0.000        0.0        0.0        0.0     0.0   
emotionality                  0.000        0.0        0.0        0.0     0.0   
extraversion                  0.000        0.0        0.0        0.0     0.0   
honest-humility               0.125        0.0        0.0        0.0     0.0   
openness to experience        0.000        0.0        0.0        0.0     0.0   

                                                  trait_score             \
paraphrase              normal         paraphrase  paraphrase     normal   
likert_scale           inverse  normal    inverse      normal     normal   
refusal_allowed        Refusal Refusal No Refusal     Refusal No Refusal   
trait                                                                      
agreeableness            0.000  0.0000        0.0       3.250      3.188   
altruism                 0.000  0.0000        0.0       4.250      4.000   
conscientiousness        0.000  0.0000        0.0       3.312      3.062   
emotionality             0.000  0.0000        0.0       3.125      3.062   
extraversion             0.000  0.0000        0.0       3.250      3.312   
honest-humility          0.125  0.0625        0.0       3.143      3.562   
openness to experience   0.000  0.0000        0.0       3.312      3.438   

                                                                      \
paraphrase                        paraphrase          normal           
likert_scale              inverse     normal inverse inverse  normal   
refusal_allowed        No Refusal No Refusal Refusal Refusal Refusal   
trait                                                                  
agreeableness               3.250      3.312   3.188   3.125   3.250   
altruism                    4.000      3.750   4.250   4.000   4.000   
conscientiousness           3.250      3.250   3.438   3.188   3.312   
emotionality                3.125      3.125   3.125   3.062   2.938   
extraversion                3.438      3.125   3.125   3.375   3.312   
honest-humility             3.500      3.562   3.438   3.643   3.133   
openness to experience      3.375      3.312   3.312   3.438   3.438   

                                   
paraphrase             paraphrase  
likert_scale              inverse  
refusal_allowed        No Refusal  
trait                              
agreeableness               3.188  
altruism                    4.250  
conscientiousness           3.375  
emotionality                3.125  
extraversion                3.188  
honest-humility             3.438  
openness to experience      3.188

In [370]:
llama_trait_stat_pivot_df

refusal_rate                                           \
paraphrase               paraphrase     normal paraphrase     normal           
likert_scale                 normal     normal     normal    inverse           
refusal_allowed          No Refusal No Refusal    Refusal No Refusal Refusal   
trait                                                                          
agreeableness                   0.0        0.0     0.1250        0.0  0.0000   
altruism                        0.0        0.0     0.0000        0.0  0.2500   
conscientiousness               0.0        0.0     0.3750        0.0  0.1875   
emotionality                    0.0        0.0     0.0000        0.0  0.1875   
extraversion                    0.0        0.0     0.0625        0.0  0.0625   
honest-humility                 0.0        0.0     0.1250        0.0  0.1250   
openness to experience          0.0        0.0     0.1875        0.0  0.0625   

                                                  trait_score             \
paraphrase                     paraphrase          paraphrase     normal   
likert_scale            normal    inverse              normal     normal   
refusal_allowed        Refusal No Refusal Refusal  No Refusal No Refusal   
trait                                                                      
agreeableness           0.0000        0.0  0.2500       3.375      3.500   
altruism                0.0000        0.0  0.0000       2.500      2.750   
conscientiousness       0.1875        0.0  0.0625       3.250      3.125   
emotionality            0.1250        0.0  0.1875       1.750      2.562   
extraversion            0.0000        0.0  0.0625       2.750      3.188   
honest-humility         0.1250        0.0  0.3125       3.188      3.438   
openness to experience  0.1250        0.0  0.1250       3.625      2.875   

                                                                         \
paraphrase             paraphrase     normal                 paraphrase   
likert_scale               normal    inverse          normal    inverse   
refusal_allowed           Refusal No Refusal Refusal Refusal No Refusal   
trait                                                                     
agreeableness               2.714      2.750   3.375   3.375      2.750   
altruism                    2.500      3.750   4.000   3.000      4.250   
conscientiousness           3.200      3.312   3.077   2.923      3.625   
emotionality                2.688      3.000   2.615   2.857      2.250   
extraversion                2.933      2.688   2.400   3.125      3.188   
honest-humility             2.714      3.000   2.786   3.643      3.562   
openness to experience      3.692      3.125   3.200   2.571      3.312   

                                
paraphrase                      
likert_scale                    
refusal_allowed        Refusal  
trait                           
agreeableness            3.417  
altruism                 5.000  
conscientiousness        3.467  
emotionality             2.538  
extraversion             3.200  
honest-humility          2.273  
openness to experience   3.786

In [363]:
gpt_subtrait_stat_pivot_df

refusal_rate             \
paraphrase                                      paraphrase     normal   
likert_scale                                        normal     normal   
refusal_allowed                                    Refusal No Refusal   
trait                  subtrait                                         
agreeableness          flexibility                     0.0        0.0   
                       forgiveness                     0.0        0.0   
                       gentleness                      0.0        0.0   
                       patience                        0.0        0.0   
altruism               altruism                        0.0        0.0   
conscientiousness      diligence                       0.0        0.0   
                       organization                    0.0        0.0   
                       perfectionism                   0.0        0.0   
                       prudence                        0.0        0.0   
emotionality           anxiety                         0.0        0.0   
                       dependence                      0.0        0.0   
                       fearfulness                     0.0        0.0   
                       sentimentality                  0.0        0.0   
extraversion           liveliness                      0.0        0.0   
                       sociability                     0.0        0.0   
                       social boldness                 0.0        0.0   
                       social self-esteem              0.0        0.0   
honest-humility        fairness                        0.5        0.0   
                       greed-avoidance                 0.0        0.0   
                       modesty                         0.0        0.0   
                       sincerity                       0.0        0.0   
openness to experience aesthetic appreciation          0.0        0.0   
                       creativity                      0.0        0.0   
                       inquisitiveness                 0.0        0.0   
                       unconventionality               0.0        0.0   

                                                                             \
paraphrase                                               paraphrase           
likert_scale                                     inverse     normal inverse   
refusal_allowed                               No Refusal No Refusal Refusal   
trait                  subtrait                                               
agreeableness          flexibility                   0.0        0.0     0.0   
                       forgiveness                   0.0        0.0     0.0   
                       gentleness                    0.0        0.0     0.0   
                       patience                      0.0        0.0     0.0   
altruism               altruism                      0.0        0.0     0.0   
conscientiousness      diligence                     0.0        0.0     0.0   
                       organization                  0.0        0.0     0.0   
                       perfectionism                 0.0        0.0     0.0   
                       prudence                      0.0        0.0     0.0   
emotionality           anxiety                       0.0        0.0     0.0   
                       dependence                    0.0        0.0     0.0   
                       fearfulness                   0.0        0.0     0.0   
                       sentimentality                0.0        0.0     0.0   
extraversion           liveliness                    0.0        0.0     0.0   
                       sociability                   0.0        0.0     0.0   
                       social boldness               0.0        0.0     0.0   
                       social self-esteem            0.0        0.0     0.0   
honest-humility        fairness                      0.0        0.0     0.0   
                       greed-avoidance

In [364]:
llama_subtrait_stat_pivot_df

refusal_rate             \
paraphrase                                      paraphrase     normal   
likert_scale                                        normal     normal   
refusal_allowed                                 No Refusal No Refusal   
trait                  subtrait                                         
agreeableness          flexibility                     0.0        0.0   
                       forgiveness                     0.0        0.0   
                       gentleness                      0.0        0.0   
                       patience                        0.0        0.0   
altruism               altruism                        0.0        0.0   
conscientiousness      diligence                       0.0        0.0   
                       organization                    0.0        0.0   
                       perfectionism                   0.0        0.0   
                       prudence                        0.0        0.0   
emotionality           anxiety                         0.0        0.0   
                       dependence                      0.0        0.0   
                       fearfulness                     0.0        0.0   
                       sentimentality                  0.0        0.0   
extraversion           liveliness                      0.0        0.0   
                       sociability                     0.0        0.0   
                       social boldness                 0.0        0.0   
                       social self-esteem              0.0        0.0   
honest-humility        fairness                        0.0        0.0   
                       greed-avoidance                 0.0        0.0   
                       modesty                         0.0        0.0   
                       sincerity                       0.0        0.0   
openness to experience aesthetic appreciation          0.0        0.0   
                       creativity                      0.0        0.0   
                       inquisitiveness                 0.0        0.0   
                       unconventionality               0.0        0.0   

                                                                             \
paraphrase                                    paraphrase     normal           
likert_scale                                      normal    inverse           
refusal_allowed                                  Refusal No Refusal Refusal   
trait                  subtrait                                               
agreeableness          flexibility                  0.50        0.0    0.00   
                       forgiveness                  0.00        0.0    0.00   
                       gentleness                   0.00        0.0    0.00   
                       patience                     0.00        0.0    0.00   
altruism               altruism                     0.00        0.0    0.25   
conscientiousness      diligence                    0.25        0.0    0.50   
                       organization                 0.75        0.0    0.25   
                       perfectionism                0.50        0.0    0.00   
                       prudence                     0.00        0.0    0.00   
emotionality           anxiety                      0.00        0.0    0.25   
                       dependence                   0.00        0.0    0.25   
                       fearfulness                  0.00        0.0    0.25   
                       sentimentality               0.00        0.0    0.00   
extraversion           liveliness                   0.25        0.0    0.00   
                       sociability                  0.00        0.0    0.25   
                       social boldness              0.00        0.0    0.00   
                       social self-esteem           0.00        0.0    0.00   
honest-humility        fairness                     0.25        0.0    0.00   
                       greed-avoidance